In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

print("=== PERSIAPAN DATA SESUAI ARAHAN DOSEN ===")
path_folder = "/content/drive/MyDrive/NLP_Project/"

# 1. LOAD 4 DATA PUBLIK (26.999 Baris)
df_publik = pd.read_csv(path_folder + "df_train_master_cleaned.csv")
df_publik = df_publik.dropna(subset=['text', 'label'])

# 2. TRAIN-TEST SPLIT (80:20) HANYA PADA DATA PUBLIK
X_pub = df_publik['text'].astype(str)
y_pub = df_publik['label']
X_train_pub, X_test_pub, y_train_pub, y_test_pub = train_test_split(
    X_pub, y_pub, test_size=0.20, random_state=42, stratify=y_pub
)
print(f"Data Publik untuk Latihan (80%): {len(X_train_pub)} baris")
print(f"Data Publik untuk Ujian Internal (20%): {len(X_test_pub)} baris")

# 3. LOAD DATA STOCKBIT (Eksternal)
path_qwen = "/content/drive/MyDrive/NLP_qwen/"
file_names = ['INET_labeled.csv', 'DEWA_labeled.csv', 'AADI_labeled.csv', 'BUMI_labeled.csv', 'ELSA_labeled.csv']
list_df_qwen = []
for file in file_names:
    df_temp = pd.read_csv(path_qwen + file)
    if 'content' in df_temp.columns and 'text' not in df_temp.columns:
        df_temp = df_temp.rename(columns={'content': 'text'})
    list_df_qwen.append(df_temp)

df_stockbit = pd.concat(list_df_qwen, ignore_index=True).dropna(subset=['text', 'label'])
X_test_mandiri = df_stockbit['text'].astype(str)
y_test_mandiri = df_stockbit['label']
print(f"Data Stockbit untuk Ujian Mandiri: {len(df_stockbit)} baris")


print("\n=== PROSES TRAINING (Murni Data Publik) ===")
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train_pub)

print("Melatih SVM...")
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_vec, y_train_pub)

print("Melatih Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train_pub)


print("TAHAP 1: UJIAN INTERNAL (20% Data Publik)")
X_test_pub_vec = vectorizer.transform(X_test_pub)

print("--- Hasil SVM (Internal) ---")
print(classification_report(y_test_pub, svm_model.predict(X_test_pub_vec)))
print("--- Hasil Random Forest (Internal) ---")
print(classification_report(y_test_pub, rf_model.predict(X_test_pub_vec)))


print("TAHAP 2: UJIAN MANDIRI (100% Data Stockbit Target)")
X_test_mandiri_vec = vectorizer.transform(X_test_mandiri)

print("--- Hasil SVM (Eksternal / Mandiri) ---")
print(classification_report(y_test_mandiri, svm_model.predict(X_test_mandiri_vec)))
print("--- Hasil Random Forest (Eksternal / Mandiri) ---")
print(classification_report(y_test_mandiri, rf_model.predict(X_test_mandiri_vec)))

=== PERSIAPAN DATA SESUAI ARAHAN DOSEN ===
Data Publik untuk Latihan (80%): 21599 baris
Data Publik untuk Ujian Internal (20%): 5400 baris
Data Stockbit untuk Ujian Mandiri: 4997 baris

=== PROSES TRAINING (Murni Data Publik) ===
Melatih SVM...
Melatih Random Forest...
TAHAP 1: UJIAN INTERNAL (20% Data Publik)
--- Hasil SVM (Internal) ---
              precision    recall  f1-score   support

     Negatif       0.81      0.79      0.80      1532
      Netral       0.78      0.77      0.77      1350
     Positif       0.86      0.88      0.87      2518

    accuracy                           0.83      5400
   macro avg       0.82      0.81      0.81      5400
weighted avg       0.82      0.83      0.83      5400

--- Hasil Random Forest (Internal) ---
              precision    recall  f1-score   support

     Negatif       0.77      0.71      0.74      1532
      Netral       0.74      0.73      0.74      1350
     Positif       0.82      0.87      0.85      2518

    accuracy         

In [ ]:
import joblib

path_folder = "/content/drive/MyDrive/NLP_Project/"

# Menyimpan Vectorizer dan Model dari skenario OOD (Murni Data Publik)
print("⏳ Menyimpan model ke Google Drive...")

try:
    # Simpan TF-IDF Vectorizer
    joblib.dump(vectorizer, path_folder + 'tfidf_vectorizer_OOD.pkl')

    # Simpan SVM Model
    joblib.dump(svm_model, path_folder + 'svm_tfidf_OOD_model.pkl')

    # Simpan Random Forest Model
    joblib.dump(rf_model, path_folder + 'rf_tfidf_OOD_model.pkl')

    print("Model SVM, Random Forest, dan TF-IDF (Skenario OOD) sudah tersimpan.")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat menyimpan: {e}")

⏳ Menyimpan model ke Google Drive...
Model SVM, Random Forest, dan TF-IDF (Skenario OOD) sudah tersimpan.
